In [8]:
import os
import json
import logging

def find_json_files(root_dir: str) -> list:
    """Find all JSON files in the given directory and its subdirectories."""
    json_files = []
    
    logging.info(f"Searching for JSON files in: {root_dir}")
    
    for root, _, files in os.walk(root_dir):
        for file in files:
            if file.endswith('.json'):
                full_path = os.path.join(root, file)
                # Get relative path from root_dir
                rel_path = os.path.relpath(full_path, root_dir)
                json_files.append(rel_path)
    
    return sorted(json_files)  # Sort for consistent output

def main():
    # Set up logging
    logging.basicConfig(level=logging.INFO,
                       format='%(asctime)s - %(levelname)s - %(message)s')
    
    # Get the labels directory path
    labels_dir = "labels"  # Adjust this path if needed
    
    try:
        # Find all JSON files
        json_files = find_json_files(labels_dir)
        
        # Print the results
        print("\nFound JSON files:")
        for file_path in json_files:
            print(file_path)
        
        print(f"\nTotal JSON files found: {len(json_files)}")
        
    except Exception as e:
        logging.error(f"Error processing directory: {str(e)}")
        return 1
    
    return json_files  # Return the files instead of an exit code

if __name__ == "__main__":
    main()  # When run as a script, just call main()

2025-02-04 07:51:54,987 - INFO - Searching for JSON files in: labels



Found JSON files:
cam_motion/arc_crane_movement/arc_clockwise/has_arc_clockwise.json
cam_motion/arc_crane_movement/arc_counterclockwise/has_arc_counterclockwise.json
cam_motion/arc_crane_movement/crane_down/has_crane_down.json
cam_motion/arc_crane_movement/crane_up/has_crane_up.json
cam_motion/camera_centric_movement/backward/has_backward_wrt_camera.json
cam_motion/camera_centric_movement/backward/only_backward_wrt_camera.json
cam_motion/camera_centric_movement/downward/has_downward_wrt_camera.json
cam_motion/camera_centric_movement/downward/only_downward_wrt_camera.json
cam_motion/camera_centric_movement/forward/has_forward_wrt_camera.json
cam_motion/camera_centric_movement/forward/only_forward_wrt_camera.json
cam_motion/camera_centric_movement/leftward/has_leftward.json
cam_motion/camera_centric_movement/leftward/only_leftward.json
cam_motion/camera_centric_movement/pan_left/has_pan_left.json
cam_motion/camera_centric_movement/pan_left/only_pan_left.json
cam_motion/camera_centric_mo

In [7]:
import os
import logging
from process_ndjson import process_ndjson_files
from video_data import VideoData

def get_video_names_from_directory(directory_path: str) -> set:
    """Get all video names from a directory."""
    video_names = set()
    for root, _, files in os.walk(directory_path):
        for file in files:
            if file.endswith('.mp4'):
                video_names.add(file)
    return video_names

def debug_video_data(video_data: VideoData, video_name: str):
    """Print debug information for a video."""
    print(f"\nDEBUG INFO FOR: {video_name}")
    print("=" * 50)
    
    # Debug Camera Motion
    print("\nCamera Motion:")
    print("-" * 20)
    try:
        motion = video_data.cam_motion
        print(f"Down motion: {motion.down}")
        print(f"No other motion (excluding down): {motion.check_if_no_motion(exclude=['down'])}")
    except AttributeError as e:
        print(f"Camera motion not set: {e}")

    # Debug Camera Setup
    print("\nCamera Setup:")
    print("-" * 20)
    try:
        setup = video_data.cam_setup
        print(f"Camera angle start: {setup.camera_angle_start}")
        print(f"Valid camera angle: {setup.camera_angle_start not in ['bird_eye_angle', 'worm_eye_angle', 'unknown']}")
    except AttributeError as e:
        print(f"Camera setup not set: {e}")

def main():
    # Configure logging
    logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')

    # Directory containing the videos to process
    videos_dir = "../datasets/CaptionAnythingtest/videos"
    if not os.path.exists(videos_dir):
        print(f"Error: Directory {videos_dir} does not exist")
        return

    # Get video names from directory
    video_names = get_video_names_from_directory(videos_dir)
    print(f"\nFound {len(video_names)} videos in directory")

    # Process NDJSON files
    ndjson_dir = "exports/ndjson"
    issues_dir = "exports/issues_ndjson"
    
    print("\nProcessing NDJSON files...")
    all_videos = process_ndjson_files(ndjson_dir, issues_dir)
    
    # Debug only the videos from our directory
    print("\nDebugging videos...")
    for video_name in video_names:
        if video_name in all_videos:
            debug_video_data(all_videos[video_name], video_name)
        else:
            print(f"\nVideo {video_name} not found in NDJSON data")

if __name__ == "__main__":
    main()


Found 30 videos in directory

Processing NDJSON files...


2025-02-04 04:09:16,919 - INFO - Processing 4527 videos (263 marked as shot transitions)...
2025-02-04 04:09:17,569 - ERROR - Error setting data for 0ade60a36915416b2fa2416dc9d8b3b9fc46313b39623ecc0a133c2dbe18fa2c.0.mp4: Subject height start should not be 'unknown' for shot types 'human', 'non_human', 'many_subject_one_focus', 'clear_subject_dynamic_size', 'clear_subject_atypical'
2025-02-04 04:09:17,579 - ERROR - Error setting data for 1a1cecb8-2476-4966-9786-3fe0dce1ec0f.24.mp4: When camera_movement is not 'major_complex', complex_motion_description must be empty.
2025-02-04 04:09:17,586 - ERROR - Error setting data for 1aa6095ed461f823bd3187040dadd0b56d7340389fa26f691bc611f2da6e7cb8.1.mp4: Focus change reason should not be 'no_change' for focus plane change
2025-02-04 04:09:17,587 - ERROR - Error setting data for 02cee54d7a90b8397b4d59f6c2b5f58ce861d12f628fc496b6f1f510d73f4b62.0.mp4: Subject height start should not be 'unknown' for shot types 'human', 'non_human', 'many_subject_one_


Debugging videos...

DEBUG INFO FOR: YBC2JaevzOI.6.4.mp4

Camera Motion:
--------------------
Down motion: None
No other motion (excluding down): False

Camera Setup:
--------------------
Camera setup not set: cam_setup has not been set

DEBUG INFO FOR: H4AZhS5WqKk.0.16.mp4

Camera Motion:
--------------------
Down motion: None
No other motion (excluding down): False

Camera Setup:
--------------------
Camera setup not set: cam_setup has not been set

DEBUG INFO FOR: lz5xvWTodyw.3.1.mp4

Camera Motion:
--------------------
Down motion: None
No other motion (excluding down): False

Camera Setup:
--------------------
Camera angle start: level_angle
Valid camera angle: True

DEBUG INFO FOR: 4847.41.1.mp4

Camera Motion:
--------------------
Down motion: None
No other motion (excluding down): False

Camera Setup:
--------------------
Camera angle start: unknown
Valid camera angle: False

DEBUG INFO FOR: -2uIa-XMJC0.6.10.mp4

Camera Motion:
--------------------
Down motion: None
No other m

In [12]:
import json

def add_and_deduplicate():
    try:
        # Read existing JSON file
        try:
            with open('donevideos/donesection.json', 'r') as f:
                video_list = json.load(f)
                if not isinstance(video_list, list):
                    print("Error: donesection.json does not contain a list")
                    return
        except FileNotFoundError:
            video_list = []
        except json.JSONDecodeError:
            print("Error: Invalid JSON format in donesection.json")
            return

        # Read add.txt file
        try:
            with open('donevideos/add.txt', 'r') as f:
                new_videos = [line.strip() for line in f if line.strip()]
            print(f"Found {len(new_videos)} videos to add from add.txt")
        except FileNotFoundError:
            print("Error: add.txt file not found")
            return

        # Combine lists and remove duplicates
        original_length = len(video_list)
        video_list.extend(new_videos)
        unique_videos = list(dict.fromkeys(video_list))
        
        # Calculate stats
        num_added = len(unique_videos) - original_length
        num_duplicates = len(video_list) - len(unique_videos)
        
        # Write back the deduplicated list
        with open('donevideos/donesection.json', 'w') as f:
            json.dump(unique_videos, f, indent=4)
            
        print(f"Added {num_added} new videos")
        print(f"Found and removed {num_duplicates} duplicate entries")
        print(f"Saved final list with {len(unique_videos)} entries")
            
    except Exception as e:
        print(f"Unexpected error: {str(e)}")

# Run the add and deduplication process
add_and_deduplicate()


Found 3893 videos to add from add.txt
Added 813 new videos
Found and removed 3080 duplicate entries
Saved final list with 3859 entries
